Second notebook for the indexing pipeline. There are somethings I wanted to change in the preprocessing step, and also generally clean up the prior notebook.

In [ ]:
# imports
import json
import re
from collections import Counter

In [17]:
# read in the raw recipes
path_raw = "../recipe_crawler/recipes.json"
with open(path_raw, 'r') as f:
    recipes = json.load(f)

# print the first few recipes
print(json.dumps(recipes[0:2], indent=4))

[
    {
        "recipe_name": "Nagi\u2019s Salad Sandwich",
        "ingredient_groups": [
            {
                "group_name": null,
                "ingredients": [
                    {
                        "amount": "1",
                        "unit": "",
                        "name": "soft bread roll",
                        "notes": ", wholemeal or white (or sliced bread, wraps)"
                    },
                    {
                        "amount": "1",
                        "unit": "",
                        "name": "small carrot",
                        "notes": ", peeled, grated using a box grater (3/4 cup lightly packed per sandwich)"
                    },
                    {
                        "amount": "1/2",
                        "unit": "",
                        "name": "ripe avocado",
                        "notes": ""
                    },
                    {
                        "amount": "1",
                        "unit

Normalise ingredient groups:
- Match casing
- Reomve excess characters

Will handle null group names with more care; if they are the only group only then can we assume they are Main.

In [ ]:
with open("../recipe_crawler/recipes.json", "r") as f:
    recipes = json.load(f)

# Count ingredient groups per recipe
group_counts = Counter()
for r in recipes:
    num_groups = len(r["ingredient_groups"])
    group_counts[num_groups] += 1

print("Distribution of ingredient groups per recipe:\n")
print(f"{'# Groups':<10} {'# Recipes':<12} {'%'}")
print("-" * 30)
for num_groups in sorted(group_counts):
    count = group_counts[num_groups]
    pct = (count / len(recipes)) * 100
    bar = "█" * int(pct / 2)
    print(f"{num_groups:<10} {count:<12} {pct:.1f}%  {bar}")

print(f"\nTotal recipes: {len(recipes)}")
print(f"Mean groups per recipe: {sum(k * v for k, v in group_counts.items()) / len(recipes):.1f}")
print(f"Range: {min(group_counts)} – {max(group_counts)}")

Distribution of ingredient groups per recipe:

# Groups   # Recipes    %
------------------------------
1          286          19.3%  █████████
2          486          32.8%  ████████████████
3          415          28.0%  ██████████████
4          207          14.0%  ██████
5          59           4.0%  █
6          18           1.2%  
7          6            0.4%  
8          1            0.1%  
9          2            0.1%  
10         1            0.1%  
13         1            0.1%  

Total recipes: 1482
Mean groups per recipe: 2.6
Range: 1 – 13


In [25]:
import json

with open("../recipe_crawler/recipes.json", "r") as f:
    recipes = json.load(f)

# Counters
only_group_null = 0      # Recipe has 1 group total, and it's None
one_of_many_null = 0     # Recipe has multiple groups, exactly 1 is None
multiple_nulls = 0       # Recipe has multiple groups, 2+ are None

for r in recipes:
    groups = r["ingredient_groups"]
    total = len(groups)
    null_count = sum(1 for g in groups if g["group_name"] is None)
    
    if null_count == 0:
        continue
    elif total == 1 and null_count == 1:
        only_group_null += 1
    elif total > 1 and null_count == 1:
        one_of_many_null += 1
    else:
        multiple_nulls += 1

print("Ingredient groups with null names:\n")
print(f"  Only group in recipe, and it's null:  {only_group_null}")
print(f"  One null among several named groups:   {one_of_many_null}")
print(f"  Multiple nulls in same recipe:         {multiple_nulls}")
print(f"  ─────────────────────────────────────")
print(f"  Total recipes with null groups:        {only_group_null + one_of_many_null + multiple_nulls}")
print(f"  Total recipes overall:                 {len(recipes)}")

Ingredient groups with null names:

  Only group in recipe, and it's null:  282
  One null among several named groups:   605
  Multiple nulls in same recipe:         8
  ─────────────────────────────────────
  Total recipes with null groups:        895
  Total recipes overall:                 1482


In [26]:
count = 0
for r in recipes:
    groups = r["ingredient_groups"]
    null_positions = [i for i, g in enumerate(groups) if g["group_name"] is None]
    
    if len(null_positions) >= 2:
        count += 1
        print("=" * 70)
        print(f"RECIPE {count}: {r['recipe_name']}")
        print(f"Total groups: {len(groups)},  Null groups: {len(null_positions)}")
        print(f"Null at positions: {null_positions}")
        print("=" * 70)
        
        for i, g in enumerate(groups):
            marker = " ← NULL" if g["group_name"] is None else ""
            print(f"\n  [{i}] group_name: {g['group_name']}{marker}")
            print(f"      group_type: {g.get('group_type', 'N/A')}")
            print(f"      ingredients ({len(g['ingredients'])}):")
            for ing in g["ingredients"]:
                print(f"        {ing['amount']} {ing['unit']} {ing['name']}".strip())
        
        print()

print(f"\nTotal: {count} recipes with multiple null groups")

RECIPE 1: One pot chicken risoni with crispy salami (orzo)
Total groups: 4,  Null groups: 2
Null at positions: [0, 3]

  [0] group_name: None ← NULL
      group_type: N/A
      ingredients (3):
1/2 tbsp olive oil
100g/ 3 oz  salami stick
2 x 250g / 8 oz  chicken breasts

  [1] group_name: Chicken spice mix:
      group_type: N/A
      ingredients (5):
1/2 tsp cooking salt / kosher salt
1/4 tsp black pepper
1/2 tsp garlic powder
3/4 tsp paprika
1/4 tsp sage powder

  [2] group_name: Creamy tomato parmesan risoni (orzo):
      group_type: N/A
      ingredients (13):
2  garlic cloves
1/2  onion
1/4 cup chardonnay or other dry white wine
1/4 cup tomato paste
1 1/4 cups risoni/orzo
400g / 14 oz  canned chickpeas
3 cups chicken stock
1/2 tsp cooking / kosher salt
1/4 tsp black pepper
3/4 cup thickened / heavy cream
1/3 cup parmesan
150g/ 5 oz  baby spinach
1/3 cup sun dried tomato

  [3] group_name: None ← NULL
      group_type: N/A
      ingredients (1):
2 tbsp roughly chopped basil

RECIPE

Recipes with multiple unnamed ingredient groups are not safe for assumption; these can be main, can be garnish, etc. Therefore, will ignore it for now.

In [32]:
def parse_amount(amount_str, unit_str):
    match = re.match(r'^([\d\s./]+)\s*([a-zA-Z]+)\s*/\s*([\d\s./]+)\s*([a-zA-Z]+)$', amount_str)

    # only keep the first match to keep it metric (pattern is that the first match is the one with metric units)
    if match:
        amount_str = match.group(1).strip()
        unit_str = match.group(2).strip()
        alt_amount = match.group(3).strip()
        alt_unit = match.group(4).strip()
        # print for debugging
        # print(f"Amount: {amount_str}, Unit: {unit_str}, Alt Amount: {alt_amount}, Alt Unit: {alt_unit}")
    
        return amount_str, unit_str, alt_amount, alt_unit
    # if dual amount not found
    return amount_str, unit_str, None, None

def standardise_ingredient_group(ingredient_groups_list):
    for ingredient_group in ingredient_groups_list:
        if ingredient_group["group_name"] is not None:
            ingredient_group["group_name"] = ingredient_group["group_name"].rstrip(":").strip().title()

        # clean each ingredient
        for ing in ingredient_group.get("ingredients", []):
            # parse dual units
            amuont, unit, alt_amount, alt_unit = parse_amount(ing["amount"], ing["unit"])
            ing["amount"] = amuont
            ing["unit"] = unit

            if alt_amount:
                ing["alt_amount"] = alt_amount
                ing["alt_unit"] = alt_unit
            
            # clean notes
            if ing.get("notes"):
                note = ing["notes"].lstrip(", ").strip()
                note = note.lstrip(" ").strip()

                # If the entire note is wrapped in parentheses, remove them
                if note.startswith("(") and note.endswith(")") and note.count("(") == 1:
                    note = note[1:-1].strip()
                
                ing["notes"] = note
        
    # handle null groups
    # find all groups in ingredient group list where group name is None
    null_groups = [g for g in ingredient_groups_list if g["group_name"] is None]
    
    if len(null_groups) == 0:
        # nothing to fix
        pass
    
    elif len(ingredient_groups_list) == 1 and len(null_groups) == 1:
        # only one group and it's null assume it's the full recipe
        null_groups[0]["group_name"] = "Full Recipe"
    
    elif len(null_groups) == 1:
        # one null among several named groups, assume it's the "Main" body
        null_groups[0]["group_name"] = "Main"
    
    return ingredient_groups_list

# normalse group names and remove leading commaas and spaces from notes
with open(path_raw, 'r') as f:
    recipes = json.load(f)

for recipe in recipes:
    standardise_ingredient_group(recipe["ingredient_groups"])

# Save
with open("recipes_standardized.json", "w") as f:
    json.dump(recipes, f, indent=2, ensure_ascii=False)

In [ ]:
# look for empty ingredients list